In [ ]:
#Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
#Load Dataset

csv_path = r'C:\Users\User\Desktop\ML Assignment\ML-Assignment\data\deliveries.csv'
df_deliveries = pd.read_csv(csv_path)

print("Dataset Loaded!")
df_deliveries.head()

In [ ]:

#Powerplay Runs

pp_runs = df_deliveries[df_deliveries['over'] <= 6] \
    .groupby(['match_id', 'inning'])['total_runs'] \
    .sum().reset_index()

pp_runs.head()

In [ ]:
#Total Runs

total_runs = df_deliveries \
    .groupby(['match_id', 'inning'])['total_runs'] \
    .sum().reset_index()

total_runs.head()

In [ ]:
#Merge data

df = pd.merge(
    pp_runs,
    total_runs,
    on=['match_id', 'inning'],
    suffixes=('_6_overs', '_total')
)

df.head()

In [ ]:
# Run rate
df['run_rate_pp'] = df['total_runs_6_overs'] / 6

# Interaction feature
df['pp_x_inning'] = df['total_runs_6_overs'] * df['inning']

print("Preprocessing Done:", df.shape)
df.head()

In [ ]:
#define x and y
X = df[['total_runs_6_overs', 'pp_x_inning']]
y = df['total_runs_total']

print(X.head())
print(y.head())

In [ ]:
#Train/Test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
#Train model

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained!")

In [ ]:
#Prediction

y_pred = model.predict(X_test)

print(y_pred[:5])

In [ ]:
#Evaluation

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)

In [ ]:
#Graph

plt.scatter(y_test, y_pred)
plt.xlabel("Actual Runs")
plt.ylabel("Predicted Runs")
plt.title("Actual vs Predicted")
plt.show()

In [ ]:
#Model Details

print("Intercept:", model.intercept_)
print("Coefficient:", model.coef_)

In [ ]:

# CONVERT TO CATEGORIES (FOR CLASSIFICATION METRICS)

def categorize_runs(x):
    if x < 140:
        return 0   # Low score
    elif x < 170:
        return 1   # Medium score
    else:
        return 2   # High score

# Apply conversion
y_test_cat = y_test.apply(categorize_runs)
y_pred_cat = pd.Series(y_test_pred).apply(categorize_runs)



#Classification Metrics
from sklearn.metrics import confusion_matrix, classification_report

#Classification Metrics
print(confusion_matrix(y_test_cat, y_pred_cat))

#Classification Report
print(classification_report(y_test_cat, y_pred_cat))

In [ ]:
#Train vs Test Accuracy

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Train R2 Score:", train_r2)
print("Test R2 Score:", test_r2)

In [ ]:
#Import required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score



#Read Data

df = pd.read_csv(r'C:\Users\User\Desktop\ML Assignment\ML-Assignment\data\matches.csv')

print("Rows & Columns:", df.shape)



#Data preprocessing

df = df.drop_duplicates()

df['winner'] = df['winner'].fillna('No Result')
df['toss_winner'] = df['toss_winner'].fillna('Unknown')
df['toss_decision'] = df['toss_decision'].fillna('field')
df['venue'] = df['venue'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')


#Create the target variable

df['result'] = (df['winner'] == df['team1']).astype(int)



#fearture selection

# Toss advantage
df['toss_adv'] = (df['toss_winner'] == df['team1']).astype(int)

df['toss_decision'] = df['toss_decision'].replace({
    'bat': 1,
    'field': 0
})

#Team performance score
win_map = df['winner'].value_counts().to_dict()

df['team1_score'] = df['team1'].map(win_map)
df['team2_score'] = df['team2'].map(win_map)

df['team1_score'] = df['team1_score'].fillna(0)
df['team2_score'] = df['team2_score'].fillna(0)

# Difference feature
df['score_gap'] = df['team1_score'] - df['team2_score']



#last 5 matches form

df = df.sort_values(by='date')

df['team1_form'] = df.groupby('team1')['result']\
    .rolling(window=5).mean().reset_index(level=0, drop=True)

df['team2_form'] = df.groupby('team2')['result']\
    .rolling(window=5).mean().reset_index(level=0, drop=True)

df['team1_form'] = df['team1_form'].fillna(0.5)
df['team2_form'] = df['team2_form'].fillna(0.5)

df['form_gap'] = df['team1_form'] - df['team2_form']


# Home condition
df['home_flag'] = (df['city'] == df['team1']).astype(int)



#select features and target

features = df[[
    'venue',
    'toss_adv',
    'toss_decision',
    'score_gap',
    'form_gap',
    'home_flag'
]].copy()

target = df['result'].copy()

# Remove NaN values and reset indices
mask = features.notna().all(axis=1)
features = features[mask].reset_index(drop=True)
target = target[mask].reset_index(drop=True)



#Encode categorical features

encoder = LabelEncoder()
features['venue'] = encoder.fit_transform(features['venue'])



#Train test split

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)



#Feature selection

sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)



#Train linear regression model

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

#Train set evaluation

train_pred = lr_model.predict(X_train)
train_pred_labels = (train_pred > 0.5).astype(int)

train_score = accuracy_score(y_train, train_pred_labels)
print("Training Accuracy:", train_score * 100)



#Test set evaluation

test_pred = lr_model.predict(X_test)
test_pred_labels = (test_pred > 0.5).astype(int)

test_score = accuracy_score(y_test, test_pred_labels)
print("Testing Accuracy:", test_score * 100)


# MODEL EVALUATION (EXTRA METRICS)

from sklearn.metrics import confusion_matrix, classification_report


#Confusion Matrix
cm = confusion_matrix(y_test, test_pred_labels)
print("Confusion Matrix:\n", cm)

#Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, test_pred_labels))

